# Loka Research Agent - Strands Version

Build a research agent about **Loka** step by step with
**[Strands Agents](https://strandsagents.com/)**.

This notebook is a **guided hands-on exercise** that you will follow throughout the workshop. The agent will be built progressively in three exercises, each adding more capabilities. You will find the following markers in the notebook to guide you:

| Marker               | Meaning                                              |
|----------------------|------------------------------------------------------|
| 🛠️ **Setup**        | Run this once to set up the environment and imports. |
| ✅ **Given**          | Code or instructions provided for you                |
| ✏️ **Your turn**     | Code you need to write to complete the exercise.     |
| 🚀 **Going further** | Optional stretch ideas if you finish early.          |
| 📚 **Hints**         | Links into the Strands docs for more information.    |

The exercises contain detailed instructions on what to do, but you are encouraged to explore and experiment. The goal is to learn by doing, so feel free to modify the code and see how it behaves. Refer to the Strands documentation for more information on the concepts and APIs used in this notebook. Make sure to switch on the `Python` toggle in the top right of documentation web page to see examples that match the code in this notebook.

## 🛠️ Setup

Run this cell once to set up the environment and imports. Make sure you have already configured an `.env` file with your **Anthropic API key** and run `uv sync` to have the project dependencies installed. If you haven't done this yet, follow the instructions in the `README.md` file.

In [1]:
import os, sys
from pathlib import Path

# Find the repo root
ROOT = next(b for b in (Path.cwd(), *Path.cwd().parents) if (b / "shared").is_dir())
sys.path.insert(0, str(ROOT / "shared"))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
assert os.getenv("ANTHROPIC_API_KEY"), "Add ANTHROPIC_API_KEY to your .env (copy .env.example)."

from strands import Agent, tool
from strands.models.anthropic import AnthropicModel
from knowledge_base import search_documents, list_topics
from website import search_website

model = AnthropicModel(
    client_args={"api_key": os.environ["ANTHROPIC_API_KEY"]},
    model_id=os.getenv("ANTHROPIC_MODEL", "claude-haiku-4-5-20251001"),
    max_tokens=1024,
    params={"temperature": 0.3},
)

print("Model ready:", model.config["model_id"])

Model ready: claude-haiku-4-5-20251001


## ✅ The knowledge base (given)

Your agent's knowledge lives in `shared/`, already written for you:
`search_documents(query)`, `list_topics()`, and `search_website(query)`. In the
exercises you'll wrap these as **tools**. Run this to see what they return (no API
key needed):

In [2]:
print(list_topics())
print("\n--- search_documents('learning') ---\n")
print(search_documents("learning"))

The Loka knowledge base covers these topics:
- AWS Innovation Partner of the Year
- Time Off and the 5/4 Friday Schedule
- Fully Remote, Work From Anywhere
- In-Person Connection Despite Being Remote
- Cutting-Edge Client Projects
- Multicultural, Global Team
- Learning and Development
- Culture of Innovation and Internal Initiatives

--- search_documents('learning') ---

## Learning and Development
One of Loka's headline goals for 2026 is to become one of the world's best learning organizations. That means real budget and real time for growth: courses, certifications, and the expectation that you keep leveling up. Learning is treated as part of the job, not a side quest.


In [3]:
print("--- search_website('services') ---\n")
print(search_website("what customers does Loka work with?"))

--- search_website('services') ---

(source: live)

[https://www.loka.com/about]
AWS, Jeff oversees Global Direct Sales at Loka, helping SMB customers unlock value through agentic AI, modernization and cloud migration. A culture of innovation Our team members live by the credo What you develop matters. We work wherever we’re most productive and take every other

[https://www.loka.com/]
17 What Fascinates Sol Rashidi? Navigating AI, balancing the possible with the practical and the art of going rogue. Jump to episode Jump to episode Blog Life at Loka • 7.8.26 Loka's Explorer Program: The Best Decision I Almost Didn't Make What an ML Engineer

[https://www.loka.com/]
Cameo. Their deep GenAI, ML and AWS expertise, paired with industry insight and exceptional customer care, makes them a standout among consultancies." Dom Scandinaro CTO , Cameo “In my 35 years of work experience, I have not come across a consulting partner like Loka. They


## Exercise 1: Basic Agent

An agent is a **model** + **tools** + **instructions**. You hand it a tool and *the model decides* when to call it.

Your job: turn the given `search_documents` function into a tool, build the agent, and run it. The system prompt is written for you.

> 💡 The tool's **docstring** is what the model reads to decide when to use it — write it for the model.

**📚 Hints**
- [Defining tools with `@tool`](https://strandsagents.com/docs/user-guide/concepts/tools/#building--loading-tools)
- [Agent API reference](https://strandsagents.com/docs/api/python/strands.agent.agent/)

### ✏️ Your turn

Build a **basic agent** that can answer questions about Loka's internal knowledge base. The model should decide when to call the tool you create.

In [4]:
from strands import Agent, tool

SYSTEM_PROMPT = """You are the Loka Research Agent, a friendly assistant that \
answers questions about Loka (the company).

- Always answer from the knowledge base via your search tool. Don't rely on prior knowledge about Loka.
- If the knowledge base has no answer, say so instead of guessing.
- Be concise, warm, and a little proud of how great Loka is to work at."""


@tool
def search_knowledge_base(query: str) -> str:
    """Search Loka's internal knowledge base for information about the company,
    its benefits, culture, and how it works.

    Args:
        query: A short natural-language description of what to look for.
    """
    return search_documents(query)


agent = Agent(model=model, tools=[search_knowledge_base], system_prompt=SYSTEM_PROMPT)


response = agent("What is Loka's time-off policy?")


Tool #1: search_knowledge_base
Great question! Loka has a fantastic time-off policy that really prioritizes work-life balance:

**The 5/4 Schedule**: Loka operates on a unique 5/4 schedule where every other Friday is off. This gives you **26 extra days off per year** on top of regular vacation time—that's roughly one long weekend every two weeks, permanently!

At Loka, work-life balance isn't just a perk—it's built right into the calendar. Pretty cool, right? 😊

### 🎯 Ask your own

In [5]:
response = agent("What kind of projects does Loka's engineering team typically work on?")


Tool #2: search_knowledge_base
Excellent question! Loka's engineering team works on some really exciting, cutting-edge projects:

**Cutting-Edge Client Projects**: The team focuses on the newest AI tools and frameworks, including:
- Generative AI
- Agent frameworks
- Modern cloud architectures

Rather than maintaining legacy systems, Loka engineers are shipping with the tools that are being showcased at major tech conferences—so you're working with the latest and greatest in the industry.

Plus, as a bonus, the team is **fully remote and global**, with teammates spread across Portugal, Brazil, Macedonia, the USA, and beyond. So you get to work on modern tech while collaborating with a genuinely multicultural team! 🌍

### 🚀 Going further

If you finished early or want to explore more, try these ideas:

- **Inspect the run.** The call returns an `AgentResult`. Look at `result.message`,
  `result.metrics` (token usage), and `result.stop_reason`.
- **Use a [built-in Strands tool](https://strandsagents.com/docs/user-guide/concepts/tools/#2-vended-tools)** instead of a custom one. For example, add `strands_tools.calculator` to your agent's `tools=[...]` and ask it something math-related. See the
  [community tools package](https://strandsagents.com/docs/user-guide/concepts/tools/community-tools-package/) for more detail.
- **Change the persona** in `SYSTEM_PROMPT` (formal? pirate?) and re-run.
- **Ask something NOT in the knowledge base** and see if the agent admits it doesn't know.

## Exercise 2: Multi-turn Agent

Two complex patterns that upgrade the basic agent built in Exercise 1:
1. **Memory** — The agent remembers the conversation and can answer follow-ups. Strands keeps the messages for you, so you don't have to manage the state yourself. The model decides when to use it.
2. **Tool selection** — The agent can choose between multiple tools. You add a second tool, and the model decides which to call.

> 💡 Use `agent.messages.clear()` to reset the memory if you want to start a new conversation.

### ✏️ Your turn

Add two new tools to the agent you built in Exercise 1 and rebuild it so it can answer questions about both Loka's internal knowledge base and its public website. The model should decide which tool to call for each question and remember the conversation for follow-up questions.

In [6]:
# Updated instructions
SYSTEM_PROMPT = """You are the Loka Research Agent, a friendly assistant that \
answers questions about Loka (the company).

Pick the right tool:
- the knowledge base tool: internal facts about working at Loka (benefits, time \
off, remote culture, awards, learning).
- the website tool: public info (services and solutions Loka offers, job \
openings, how it positions itself).

It's a conversation: use earlier turns to resolve follow-ups ("that", "there").
If neither source has the answer, say so. Be concise, warm, and proud of Loka."""


@tool
def list_knowledge_base_topics() -> str:
    """List the topics available in Loka's knowledge base. Use for overview or
    "what can you tell me about" questions rather than specific ones."""
    return list_topics()


@tool
def search_loka_website(query: str) -> str:
    """Search Loka's public website (loka.com) for what the company presents
    publicly: the services and solutions it offers, open roles, podcast/blog,
    and positioning. Use for public/marketing/services questions, as opposed to
    the internal facts in search_knowledge_base.

    Args:
        query: A short natural-language description of what to look for.
    """
    return search_website(query)


agent = Agent(
    model=model,
    tools=[search_knowledge_base, list_knowledge_base_topics, search_loka_website],
    system_prompt=SYSTEM_PROMPT,
)
# No -- memory doesn't need to be indicated explicitly; reusing one `Agent`
# instance keeps the whole conversation's history automatically.

### ✅ Run a multi-turn conversation

Run the following three questions in order. Each prints just the answer. Notice how different questions use different tools, and that the third question works only because the agent remembers the second.

In [7]:
response = agent("What can you tell me about Loka?")


Tool #1: list_knowledge_base_topics

Tool #2: search_loka_website
Great question! Here's what I can tell you about **Loka**:

## What Loka Does
Loka is a forward-thinking technology company that helps bring ideas to market faster. We advise and implement solutions in areas that make a material impact. Our core services include:
- **GenAI & ML** – Artificial intelligence and machine learning solutions
- **Data & Analytics** – Data-driven insights and analytics
- **DevOps & Migrations** – Cloud modernization and infrastructure work

## Our Culture & Team
- **Fully Remote, Work From Anywhere** – We're a distributed, global team that works wherever people are most productive
- **Multicultural, Global Team** – Diverse perspectives from around the world
- **Culture of Innovation** – Our team lives by the credo: *"What you develop matters"*
- **In-Person Connection** – Despite being remote, we maintain meaningful in-person connections

## Benefits & Work Life
- **5/4 Friday Schedule** – Gene

In [8]:
print("tools used so far:", list(response.metrics.tool_metrics))

tools used so far: ['list_knowledge_base_topics', 'search_loka_website']


In [9]:
response = agent("What has AWS recognized Loka for?")


Tool #3: search_knowledge_base
AWS recognized Loka as **Innovation Partner of the Year** – which is a significant honor! 

Here's what that means: When Amazon Web Services wants to highlight someone doing genuinely novel and cutting-edge work on their cloud platform, they point to Loka. 

This recognition reflects that:
- **We work with cutting-edge AWS services** – Our client projects leverage advanced services like Bedrock, SageMaker, and Lambda
- **We have direct partner access** – This enables us to build innovative solutions at the forefront of what's possible on AWS
- **We're doing genuinely novel things** – We're not just using AWS; we're pushing the boundaries of what can be done with it

It's a testament to Loka's commitment to innovation and our ability to deliver real, impactful solutions for our clients using the latest cloud and AI technologies.

In [10]:
print("tools used so far:", list(response.metrics.tool_metrics))  # expect: the knowledge base

tools used so far: ['list_knowledge_base_topics', 'search_loka_website', 'search_knowledge_base']


In [11]:
response = agent("How would that show up in the projects engineers work on?") # requires memory

Based on what I know about Loka, here's how that AWS recognition translates into the actual work engineers do:

**Cutting-Edge Technologies in Real Projects:**
- Engineers work on **cutting-edge client projects** that run on advanced AWS services like Bedrock (generative AI), SageMaker (machine learning), and Lambda (serverless computing)
- This means you're not maintaining legacy systems—you're building with the latest, most innovative tools available

**Direct Partner Access:**
- Having direct AWS partner access means engineers get closer collaboration with AWS, potentially early access to new features, better support, and deeper technical resources
- This translates to being able to solve problems in more sophisticated ways

**GenAI & ML Focus:**
- A lot of Loka's work centers on **GenAI & ML** solutions, which are exactly the kinds of innovative, forward-thinking projects that earned the AWS award
- Engineers get to work on AI and machine learning implementations that are genuinely

In [12]:
print("tools used so far:", list(response.metrics.tool_metrics))

tools used so far: ['list_knowledge_base_topics', 'search_loka_website', 'search_knowledge_base']


In [13]:
print(f"Memory holds {len(agent.messages)} messages — Strands kept them for us.\n")

Memory holds 10 messages — Strands kept them for us.



### 🎯 Ask your own


In [14]:
agent.messages.clear()

In [15]:
response = agent("Is Loka hiring right now, and what's the interview process like?")
print("\n\ntools used so far:", list(response.metrics.tool_metrics))

I'll search for information about current job openings and the interview process at Loka.
Tool #4: search_loka_website

Tool #5: search_knowledge_base
Great news! **Yes, Loka is actively hiring.** According to their careers page, new positions open up almost weekly, and they're currently seeking talented people across various roles including:

- **Senior Machine Learning Engineer** (LatAm, Western Europe, or Eastern Europe)
- **Project Management** roles
- And other positions in Engineering and beyond

You can check out all open job openings on their [careers page](https://www.loka.com/careers).

As for the **interview process**, I wasn't able to find specific details about that in Loka's internal knowledge base. For detailed information about what to expect in interviews, I'd recommend:
1. Checking the specific job posting you're interested in
2. Reaching out to Loka's recruitment team directly through their careers page
3. Checking their blog or podcast for any insights from team mem

### 🚀 Going further (optional)

If you finished early or want to explore more, try these ideas:

- **Try an ambiguous follow-up** and see which tool it picks.
- **Peek at memory.** Print `agent.messages` to see exactly how Strands stores the conversation.
- **Cap the history** with a
  [conversation manager](https://strandsagents.com/docs/user-guide/concepts/agents/conversation-management/)
  (e.g. a sliding window) and see what changes.
- **Modify the system prompt** to change the way the agent handles follow-up questions.
- **Play with the model parameters** (temperature, max_tokens) and see how it affects the agent's behavior.

## Exercise 3: Open-ended Reasoning

Strands has a model-driven approach to agent building. Because of this, you can build an agent that can do **open-ended research** without writing any loops or planners. The model itself decides how to reason, which tools to call, and when to stop.

**📚 Hints**
- [Agent API reference (the agent loop)](https://strandsagents.com/docs/api/python/strands.agent.agent/)
- [Tools overview](https://strandsagents.com/docs/user-guide/concepts/tools/)

### ✏️ Your turn

In [17]:
# Instructions for open-ended research (given):
SYSTEM_PROMPT = """You are the Loka Research Agent, a thorough assistant that \
answers questions about Loka (the company).

For open-ended questions:
- Break the question into the separate things you need to find out.
- Gather evidence with the tools, drawing on BOTH sources (the internal \
knowledge base AND the public website). Search multiple times; search again if \
you're missing something.
- Only once you have enough evidence, synthesize one well-structured answer.

Base every claim on the evidence you gathered; if something isn't covered, say \
so. Be warm, concrete, and proud of Loka."""


agent = Agent(
    model=model,
    tools=[search_knowledge_base, list_knowledge_base_topics, search_loka_website],
    system_prompt=SYSTEM_PROMPT,
)

result = agent(
    "I'm an ML engineer deciding whether to apply to Loka. Look into the kind of "
    "work and services the company does publicly, and cross-check it against the "
    "internal benefits and culture. Then make an honest, evidence-based case for "
    "whether it's a good fit for someone who wants cutting-edge AI work with "
    "strong work-life balance."
)

I'll help you make an informed decision. Let me gather information about Loka's public positioning, the actual work, and the internal culture and benefits.
Tool #1: search_loka_website

Tool #2: search_loka_website

Tool #3: search_knowledge_base

Tool #4: search_knowledge_base

Tool #5: list_knowledge_base_topics
Good start. Let me dig deeper into the cutting-edge work and the specific ML/AI projects:
Tool #6: search_knowledge_base

Tool #7: search_loka_website

Tool #8: search_knowledge_base

Tool #9: search_loka_website
Excellent. Let me get a bit more detail on compensation, benefits, and any potential trade-offs:
Tool #10: search_knowledge_base

Tool #11: search_knowledge_base
Now let me synthesize what I've found into an honest assessment:

## The Case for Loka: An Evidence-Based Assessment

**The Good News: This Could Be a Strong Fit**

Loka genuinely delivers on both fronts you care about—cutting-edge AI work *and* work-life balance. Here's what the evidence shows:

### ✅ Cutti

### ✅ How much work did that take?


In [18]:
print("Reasoning cycles:", result.metrics.cycle_count)
for name, t in result.metrics.tool_metrics.items():
    print(f"  {name}: {t.call_count} call(s)")
print("Tokens:", result.metrics.accumulated_usage)

Reasoning cycles: 4
  search_knowledge_base: 6 call(s)
  list_knowledge_base_topics: 1 call(s)
  search_loka_website: 4 call(s)
Tokens: {'inputTokens': 10444, 'outputTokens': 1431, 'totalTokens': 11875}


### 🎯 Ask your own


In [20]:
agent.messages.clear()
result = agent(
    "How does Loka support engineers who want to grow technically, and is that "
    "reflected in the kind of client work they get exposed to?"
)

I'll search for information about Loka's approach to engineer growth and the types of client work available.
Tool #17: search_knowledge_base

Tool #18: search_knowledge_base

Tool #19: search_loka_website

Tool #20: search_loka_website
Let me search for more specific information about the types of technical projects and client work.
Tool #21: search_knowledge_base

Tool #22: search_knowledge_base

Tool #23: search_loka_website
Perfect! I now have a clear picture. Let me synthesize this into a comprehensive answer.

## How Loka Supports Technical Growth

Loka takes engineer development seriously as a core company value. Here's what they do:

**Structured Learning Investment:**
- **Real budget and real time**: Learning isn't squeezed in around work—it's treated as part of the job itself, not a side quest
- **Courses, certifications, and continuous leveling up**: There's an explicit expectation that engineers keep growing technically
- **2026 goal**: Becoming one of the world's best learn

### 🚀 Going further (optional)

- **Compare phrasings.** Ask the same thing three ways and compare the show_tool_calls` counts — does a more specific question mean fewer searches?
- **New tool.** Add a new custom tool with a capability you would like to see the model use, and see if it can reason about when to call it.
- **Try to interrupt the model.** Is the Strands agent able to ask for human input natively to continue the research?
- **Tune the model.** Change `max_tokens` or `temperature` where `model` is defined and observe the difference.
- **Find the gaps.** Ask something the sources only partly cover and see how honestly it flags what it doesn't know.

## Conclusion

You built a research agent that can answer questions about Loka, starting with a single lookup and evolving to open-ended research across multiple sources. The model itself handled the reasoning, tool selection, and memory management, allowing you to focus on defining the tools and instructions.

The code you wrote was minimal, yet the agent's capabilities grew significantly with each exercise. Strands abstracts away the complexity of managing state and control flow, letting the model do the heavy lifting. Prompt engineering and tool design were the main levers you used to shape the agent's behavior.

#### Questions for further reflection:
- How did the model's behavior change as you added more tools and memory? Did the latency and number of tool calls increase, and if so, was it worth it for the richer answers?
- How does this agent compare to a traditional programmatic approach where you would explicitly code the reasoning and control flow? What are the trade-offs in terms of flexibility, maintainability, and performance?
- When would you *want* to take that control back and make the steps explicit?